# 06 — Hybrid Ranker

Findings so far that shape this design:
- Content-based (SBERT/FAISS) works well — validated qualitatively in notebook 04.
- Interaction data is sparse (long-tail, median ~1-2 interactions/user).
- SVD's Precision@10 (0.77) is inflated by the 88.7% positive-rating base rate, not genuine personalization — RMSE 1.22 / MAE 0.75 tell the more honest story.

Design: content-based is the primary signal. SVD contributes but is weighted low. Popularity acts as a fallback/tiebreaker for cold-start users where SVD has nothing to say. Combination is **learned** (logistic regression), not a hand-picked weight — same standard applied elsewhere (DataQ's unvalidated 0.7/0.3 was flagged as a weakness; not repeating that here).

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import faiss
import pickle

In [ ]:
PROCESSED_DIR = Path.cwd().parent / "datasets" / "processed"
EMBEDDINGS_DIR = Path.cwd().parent / "datasets" / "embeddings"
MODELS_DIR = Path.cwd().parent / "models"
RAW_DIR = Path.cwd().parent / "datasets"

df = pd.read_parquet(PROCESSED_DIR / "feature_engineered_recipes.parquet")
embeddings = np.load(EMBEDDINGS_DIR / "sbert_embeddings.npy")
faiss_index = faiss.read_index(str(EMBEDDINGS_DIR / "faiss_index.bin"))
interactions = pd.read_csv(RAW_DIR / "RAW_interactions.csv")

with open(MODELS_DIR / "svd_model.pkl", "rb") as f:
    svd_model = pickle.load(f)

print(df.shape, embeddings.shape)

## 1. Popularity score

Bayesian-adjusted average rating (not raw mean — a recipe with one 5-star rating shouldn't outrank one with 200 ratings averaging 4.8). Standard IMDB-style weighted rating formula.

In [ ]:
valid_recipe_ids = set(df.loc[df["source"] == "food.com", "id"].dropna())
interactions_filtered = interactions[interactions["recipe_id"].isin(valid_recipe_ids)].copy()

stats = interactions_filtered.groupby("recipe_id").agg(
    avg_rating=("rating", "mean"), n_ratings=("rating", "count")
)

C = stats["avg_rating"].mean()          # global mean rating
m = stats["n_ratings"].quantile(0.60)    # minimum ratings threshold — tune if needed

stats["popularity_score"] = (
    (stats["n_ratings"] / (stats["n_ratings"] + m)) * stats["avg_rating"] +
    (m / (stats["n_ratings"] + m)) * C
)

print(f"Global mean rating C={C:.3f}, min-ratings threshold m={m:.1f}")
stats.sort_values("popularity_score", ascending=False).head(5)

## 2. Build training data for the ranker

For each (user, recipe) interaction in the filtered set, compute three scores:
- **content_score**: cosine similarity between this recipe and the user's other liked recipes (proxy: similarity to their highest-rated recipe)
- **svd_score**: SVD's predicted rating
- **popularity_score**: from step 1

Label: whether the actual rating was >= 4 (relevant) or not — this becomes a binary classification problem the ranker learns to solve.

In [ ]:
id_to_row = {rid: i for i, rid in enumerate(df.loc[df["source"] == "food.com", "id"])}
food_row_indices = df.index[df["source"] == "food.com"].tolist()
id_to_embedding_idx = {df.loc[i, "id"]: i for i in food_row_indices}

# For each user, find their highest-rated recipe as a reference point for content similarity
user_top_recipe = (
    interactions_filtered.sort_values("rating", ascending=False)
    .drop_duplicates(subset="user_id", keep="first")
    .set_index("user_id")["recipe_id"]
)

print(f"Reference recipe available for {len(user_top_recipe)} users")

In [ ]:
# Sample down for tractability — computing content similarity per interaction is the expensive part
SAMPLE_SIZE = 30000
sample = interactions_filtered.sample(n=min(SAMPLE_SIZE, len(interactions_filtered)), random_state=42)

rows = []
for r in sample.itertuples():
    ref_recipe_id = user_top_recipe.get(r.user_id)
    if ref_recipe_id is None or ref_recipe_id not in id_to_embedding_idx or r.recipe_id not in id_to_embedding_idx:
        continue

    ref_idx = id_to_embedding_idx[ref_recipe_id]
    target_idx = id_to_embedding_idx[r.recipe_id]
    content_score = float(np.dot(embeddings[ref_idx], embeddings[target_idx]))  # already normalized -> cosine sim

    svd_pred = svd_model.predict(r.user_id, r.recipe_id).est

    pop_score = stats.loc[r.recipe_id, "popularity_score"] if r.recipe_id in stats.index else C

    rows.append({
        "user_id": r.user_id,
        "recipe_id": r.recipe_id,
        "content_score": content_score,
        "svd_score": svd_pred,
        "popularity_score": pop_score,
        "label": int(r.rating >= 4),
    })

ranker_data = pd.DataFrame(rows)
print(ranker_data.shape)
ranker_data.head()

## 3. Train the ranker

Logistic regression — simple, interpretable coefficients (you can literally read off how much weight each signal gets, unlike a hand-picked combination).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

X = ranker_data[["content_score", "svd_score", "popularity_score"]]
y = ranker_data["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

ranker = LogisticRegression(class_weight="balanced")  # balanced, since 88.7% positive is a real class imbalance
ranker.fit(X_train, y_train)

y_pred_proba = ranker.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Ranker AUC: {auc:.4f}")
print(f"\nLearned weights: {dict(zip(X.columns, ranker.coef_[0]))}")
print(f"\n{classification_report(y_test, ranker.predict(X_test))}")

### Reading the learned weights

Whichever coefficient is largest (after considering feature scale — these three scores aren't on identical scales, so also sanity-check with standardized coefficients if the raw ones are misleading) tells you which signal the data actually prefers, rather than assuming content-based dominates just because it looked good qualitatively. Report the AUC and the three coefficients back.

## 4. End-to-end recommend function using the hybrid ranker

In [ ]:
def hybrid_recommend(user_id, top_n: int = 10, candidate_pool: int = 50) -> pd.DataFrame:
    """Recommend recipes for a user by combining content, SVD, and popularity via the learned ranker."""
    ref_recipe_id = user_top_recipe.get(user_id)

    if ref_recipe_id is None or ref_recipe_id not in id_to_embedding_idx:
        # Cold-start user: no interaction history to anchor content similarity — fall back to popularity
        top_pop = stats.sort_values("popularity_score", ascending=False).head(top_n)
        return df[df["id"].isin(top_pop.index)][["title", "source"]].assign(
            reason="cold_start_popularity_fallback"
        )

    ref_idx = id_to_embedding_idx[ref_recipe_id]
    query_vec = embeddings[ref_idx:ref_idx+1]
    _, candidate_indices = faiss_index.search(query_vec, candidate_pool)

    candidates = []
    for idx in candidate_indices[0]:
        row = df.iloc[idx]
        if row["source"] != "food.com" or pd.isna(row["id"]):
            content_score = float(np.dot(embeddings[ref_idx], embeddings[idx]))
            svd_score = C  # no SVD signal possible outside food.com/interactions
            pop_score = C
        else:
            recipe_id = row["id"]
            content_score = float(np.dot(embeddings[ref_idx], embeddings[idx]))
            svd_score = svd_model.predict(user_id, recipe_id).est
            pop_score = stats.loc[recipe_id, "popularity_score"] if recipe_id in stats.index else C

        candidates.append({
            "idx": idx, "title": row["title"], "source": row["source"],
            "content_score": content_score, "svd_score": svd_score, "popularity_score": pop_score,
        })

    cand_df = pd.DataFrame(candidates)
    cand_df["ranker_score"] = ranker.predict_proba(
        cand_df[["content_score", "svd_score", "popularity_score"]]
    )[:, 1]

    return cand_df.sort_values("ranker_score", ascending=False)[
        ["title", "source", "content_score", "svd_score", "popularity_score", "ranker_score"]
    ].head(top_n)

In [ ]:
# Test with a real user_id from the interactions data
sample_user = interactions_filtered["user_id"].iloc[0]
hybrid_recommend(sample_user)

## 5. Save the ranker

In [ ]:
with open(MODELS_DIR / "hybrid_ranker.pkl", "wb") as f:
    pickle.dump({
        "ranker": ranker,
        "popularity_stats": stats,
        "global_mean_rating": C,
    }, f)

print(f"Saved hybrid ranker -> {MODELS_DIR / 'hybrid_ranker.pkl'}")

## Next: `07_evaluation.ipynb`

You have per-component numbers (content-based qualitative check, SVD RMSE/MAE/Precision@10, ranker AUC) but no end-to-end evaluation of the full hybrid pipeline against a proper held-out test set with a fixed protocol. That's the last thing missing before this is genuinely interview-defensible — worth doing before touching the API layer, per the earlier project-status discussion.